# RealWaste YOLO Box-Prompted SAM Segmentation Preparation

Standalone Kaggle notebook for preparing RealWaste into a normalized COCO segmentation dataset.

The pipeline loads a one-class YOLO26m detector, detects trash bounding boxes for each image, prompts SAM2 or SAM3 with those boxes, saves polygon masks in annotations.json, visualizes prepared samples, and creates a downloadable zip artifact.


In [ ]:
from pathlib import Path
import importlib.util
from importlib.metadata import PackageNotFoundError, version as package_version
import json
import os
import shutil
import subprocess
import sys

INSTALL_MISSING_PACKAGES = True
MIN_ULTRALYTICS_VERSION = (8, 3, 237)


def version_tuple(value: str) -> tuple[int, int, int]:
    parts = []
    for chunk in value.replace("-", ".").split("."):
        digits = "".join(ch for ch in chunk if ch.isdigit())
        parts.append(int(digits) if digits else 0)
        if len(parts) == 3:
            break
    while len(parts) < 3:
        parts.append(0)
    return tuple(parts[:3])


if INSTALL_MISSING_PACKAGES:
    install_specs = []
    try:
        installed_ultralytics = package_version("ultralytics")
        if version_tuple(installed_ultralytics) < MIN_ULTRALYTICS_VERSION:
            install_specs.append("ultralytics>=8.3.237")
    except PackageNotFoundError:
        install_specs.append("ultralytics>=8.3.237")

    required_packages = [
        ("PIL", "pillow>=10.0"),
        ("matplotlib", "matplotlib>=3.7"),
    ]
    install_specs.extend(package for module, package in required_packages if importlib.util.find_spec(module) is None)
    if install_specs:
        print("Installing packages:", install_specs)
        subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "-U", *install_specs])
    else:
        print("Dependencies are already installed.")


In [ ]:
from PIL import Image


def env_bool(name: str, default: bool) -> bool:
    value = os.getenv(name)
    if value is None:
        return default
    return value.strip().lower() in {"1", "true", "yes", "y", "on"}


IS_KAGGLE = Path("/kaggle").exists()
INPUT_ROOT = Path("/kaggle/input") if IS_KAGGLE else Path("data/raw")
WORKING_DIR = Path("/kaggle/working") if IS_KAGGLE else Path(".")

# Leave as None to auto-discover a folder containing RealWaste class directories.
REALWASTE_ROOT_OVERRIDE = os.getenv("REALWASTE_ROOT_OVERRIDE") or None

# Detection model: point this at your trained one-class trash YOLO26m weights in /kaggle/input when needed.
DETECT_MODEL_PATH = os.getenv("DETECT_MODEL_PATH", "yolo26m.pt")
DETECT_CONF = float(os.getenv("DETECT_CONF", "0.25"))
DETECT_IOU = float(os.getenv("DETECT_IOU", "0.45"))
DETECT_IMGSZ = int(os.getenv("DETECT_IMGSZ", "640"))
DETECT_MAX_DET = int(os.getenv("DETECT_MAX_DET", "20"))
DETECT_BOX_SELECTION = os.getenv("DETECT_BOX_SELECTION", "all").lower().strip()  # all, largest, or confidence
MAX_BOXES_PER_IMAGE_ENV = os.getenv("MAX_BOXES_PER_IMAGE", "").strip()
MAX_BOXES_PER_IMAGE = int(MAX_BOXES_PER_IMAGE_ENV) if MAX_BOXES_PER_IMAGE_ENV else None

# SAM model: default is high-accuracy SAM2.1 large. Set this to a local sam3.pt path to use SAM3.
SAM_MODEL_PATH = os.getenv("SAM_MODEL_PATH", "sam2.1_l.pt")
SAM_IMGSZ = int(os.getenv("SAM_IMGSZ", "1024"))
SAM_RETINA_MASKS = env_bool("SAM_RETINA_MASKS", True)
ALLOW_ULTRALYTICS_AUTO_DOWNLOAD = env_bool("ALLOW_ULTRALYTICS_AUTO_DOWNLOAD", True)

FALLBACK_TO_FULL_IMAGE_BOX = env_bool("FALLBACK_TO_FULL_IMAGE_BOX", True)
BOX_FALLBACK_TO_RECTANGLE = env_bool("BOX_FALLBACK_TO_RECTANGLE", True)
FORCE_REGENERATE_ANNOTATIONS = env_bool("FORCE_REGENERATE_ANNOTATIONS", False)
SAVE_CACHE_EVERY = int(os.getenv("SAVE_CACHE_EVERY", "25"))
MAX_IMAGES_ENV = os.getenv("MAX_IMAGES", "").strip()
MAX_IMAGES = int(MAX_IMAGES_ENV) if MAX_IMAGES_ENV else None

OUTPUT_DIR = WORKING_DIR / "data" / "normalized" / "realwaste"
CACHE_PATH = OUTPUT_DIR / "yolo_sam_predictions_cache.json"
ANNOTATIONS_PATH = OUTPUT_DIR / "annotations.json"
SUMMARY_PATH = OUTPUT_DIR / "realwaste_prepare_summary.json"

IMAGE_EXTENSIONS = {".jpg", ".jpeg", ".png", ".bmp", ".webp"}
EXPECTED_CLASSES = {
    "Cardboard",
    "Food Organics",
    "Glass",
    "Metal",
    "Miscellaneous Trash",
    "Paper",
    "Plastic",
    "Textile Trash",
    "Vegetation",
}
CATEGORIES = [{"id": 1, "name": "trash"}]

print("IS_KAGGLE:", IS_KAGGLE)
print("INPUT_ROOT:", INPUT_ROOT)
print("WORKING_DIR:", WORKING_DIR)
print("OUTPUT_DIR:", OUTPUT_DIR)
print("DETECT_MODEL_PATH:", DETECT_MODEL_PATH)
print("DETECT_CONF:", DETECT_CONF, "DETECT_IOU:", DETECT_IOU, "DETECT_IMGSZ:", DETECT_IMGSZ)
print("DETECT_BOX_SELECTION:", DETECT_BOX_SELECTION, "MAX_BOXES_PER_IMAGE:", MAX_BOXES_PER_IMAGE)
print("SAM_MODEL_PATH:", SAM_MODEL_PATH)
print("SAM_IMGSZ:", SAM_IMGSZ, "SAM_RETINA_MASKS:", SAM_RETINA_MASKS)
print("FALLBACK_TO_FULL_IMAGE_BOX:", FALLBACK_TO_FULL_IMAGE_BOX)
print("BOX_FALLBACK_TO_RECTANGLE:", BOX_FALLBACK_TO_RECTANGLE)

if INPUT_ROOT.exists():
    print()
    print("Input roots:")
    for path in sorted(INPUT_ROOT.iterdir()):
        print(" -", path)


In [ ]:
# Load YOLO detector and SAM segmenter before the image-processing loop starts.
from ultralytics import YOLO, SAM
from ultralytics.utils.downloads import ASSETS_URL, GITHUB_ASSETS_NAMES, safe_download

if DETECT_BOX_SELECTION not in {"all", "largest", "confidence"}:
    raise ValueError("DETECT_BOX_SELECTION must be one of: all, largest, confidence")

DETECT_MODEL_INSTANCE = None
SAM_MODEL_INSTANCE = None
DETECT_MODEL_PATH_RESOLVED = None
SAM_MODEL_PATH_RESOLVED = None


def is_simple_weight_name(model_name: str) -> bool:
    return model_name.endswith((".pt", ".pth")) and "/" not in model_name and "\\" not in model_name


def resolve_model_path(model_value: str | Path, *, model_kind: str, weights_subdir: str, env_var_name: str) -> str | Path:
    model_path = Path(str(model_value))
    if model_path.exists():
        return model_path.resolve()

    model_name = str(model_value).strip()
    if model_kind.lower() == "sam" and model_name.lower() == "sam3.pt":
        raise FileNotFoundError(
            "SAM3 weights are not automatically downloadable. Request/download sam3.pt, attach it as a Kaggle input, "
            f"then set {env_var_name} to that local file path."
        )

    if model_name in GITHUB_ASSETS_NAMES:
        weights_dir = WORKING_DIR / "models" / weights_subdir
        weights_dir.mkdir(parents=True, exist_ok=True)
        resolved = weights_dir / model_name
        if not resolved.exists():
            print(f"Downloading {model_kind} model {model_name} to {resolved}...")
            safe_download(
                url=f"{ASSETS_URL}/{model_name}",
                file=resolved,
                min_bytes=1_000_000,
                exist_ok=True,
                progress=True,
            )
        else:
            print(f"{model_kind} model already exists: {resolved}")
        return resolved

    if ALLOW_ULTRALYTICS_AUTO_DOWNLOAD and is_simple_weight_name(model_name):
        print(f"{model_kind} model is not local; Ultralytics will resolve or download it on load: {model_name}")
        return model_name

    raise FileNotFoundError(
        f"{model_kind} model '{model_value}' was not found. Set {env_var_name} to a local .pt/.pth path under /kaggle/input."
    )


DETECT_MODEL_PATH_RESOLVED = resolve_model_path(
    DETECT_MODEL_PATH,
    model_kind="YOLO detector",
    weights_subdir="yolo",
    env_var_name="DETECT_MODEL_PATH",
)
SAM_MODEL_PATH_RESOLVED = resolve_model_path(
    SAM_MODEL_PATH,
    model_kind="SAM",
    weights_subdir="sam",
    env_var_name="SAM_MODEL_PATH",
)

print("Loading YOLO detector:", DETECT_MODEL_PATH_RESOLVED)
DETECT_MODEL_INSTANCE = YOLO(str(DETECT_MODEL_PATH_RESOLVED))
print("Loading SAM segmenter:", SAM_MODEL_PATH_RESOLVED)
SAM_MODEL_INSTANCE = SAM(str(SAM_MODEL_PATH_RESOLVED))
print("Models are ready.")


In [ ]:
def discover_realwaste_root(input_root: Path = INPUT_ROOT, override: str | Path | None = REALWASTE_ROOT_OVERRIDE) -> Path:
    if override:
        root = Path(override)
        if not root.exists():
            raise FileNotFoundError(f"REALWASTE_ROOT_OVERRIDE does not exist: {root}")
        return root

    search_roots = [input_root]
    if not IS_KAGGLE:
        search_roots.extend([Path("data/raw"), Path("../data/raw")])

    candidates = []
    for root in search_roots:
        if not root.exists():
            continue
        paths = [root]
        paths.extend(path for path in root.glob("*") if path.is_dir())
        paths.extend(path for path in root.glob("*/*") if path.is_dir())
        for path in paths:
            try:
                child_dirs = {child.name for child in path.iterdir() if child.is_dir()}
            except OSError:
                continue
            score = len(child_dirs & EXPECTED_CLASSES)
            if score >= 5:
                candidates.append((score, -len(path.parts), path))

    if not candidates:
        raise FileNotFoundError("Could not find a RealWaste root containing the expected class folders.")

    candidates.sort(reverse=True)
    return candidates[0][2]


def safe_file_component(value: str) -> str:
    return "_".join(value.replace("-", "_").split())


def load_json(path: Path) -> dict:
    if not path.exists():
        return {}
    with path.open("r", encoding="utf-8") as handle:
        return json.load(handle)


def save_json(payload: dict, path: Path) -> None:
    path.parent.mkdir(parents=True, exist_ok=True)
    with path.open("w", encoding="utf-8") as handle:
        json.dump(payload, handle, indent=2)


def polygon_area(segmentation: list[float]) -> float:
    if len(segmentation) < 6:
        return 0.0
    points = list(zip(segmentation[0::2], segmentation[1::2]))
    area = 0.0
    for index, (x1, y1) in enumerate(points):
        x2, y2 = points[(index + 1) % len(points)]
        area += (x1 * y2) - (x2 * y1)
    return abs(area) / 2.0


def bbox_area(bbox: list[float]) -> float:
    return max(0.0, float(bbox[2])) * max(0.0, float(bbox[3]))


def bbox_from_segmentation(segmentation: list[float]) -> list[float]:
    xs = segmentation[0::2]
    ys = segmentation[1::2]
    min_x = min(xs)
    min_y = min(ys)
    max_x = max(xs)
    max_y = max(ys)
    return [min_x, min_y, max_x - min_x, max_y - min_y]


def clamp_xyxy(box: list[float], width: int, height: int) -> list[float]:
    x1, y1, x2, y2 = [float(v) for v in box]
    max_x = float(max(width - 1, 1))
    max_y = float(max(height - 1, 1))
    x1 = max(0.0, min(max_x, x1))
    y1 = max(0.0, min(max_y, y1))
    x2 = max(0.0, min(max_x, x2))
    y2 = max(0.0, min(max_y, y2))
    if x2 < x1:
        x1, x2 = x2, x1
    if y2 < y1:
        y1, y2 = y2, y1
    return [x1, y1, x2, y2]


def xyxy_to_xywh(box: list[float], width: int, height: int) -> list[float]:
    x1, y1, x2, y2 = clamp_xyxy(box, width, height)
    return [x1, y1, x2 - x1, y2 - y1]


def full_image_xyxy(width: int, height: int) -> list[float]:
    return [0.0, 0.0, float(max(width - 1, 1)), float(max(height - 1, 1))]


def rectangle_segmentation_from_xyxy(box: list[float], width: int, height: int) -> list[float]:
    x1, y1, x2, y2 = clamp_xyxy(box, width, height)
    return [x1, y1, x2, y1, x2, y2, x1, y2]


def clamp_polygon(segmentation: list[float], width: int, height: int) -> list[float]:
    max_x = float(max(width - 1, 1))
    max_y = float(max(height - 1, 1))
    clamped = []
    for index in range(0, len(segmentation) - 1, 2):
        x = max(0.0, min(max_x, float(segmentation[index])))
        y = max(0.0, min(max_y, float(segmentation[index + 1])))
        clamped.extend([x, y])
    return clamped


def tensor_to_list(value):
    if value is None:
        return []
    if hasattr(value, "detach"):
        value = value.detach()
    if hasattr(value, "cpu"):
        value = value.cpu()
    if hasattr(value, "numpy"):
        return value.numpy().tolist()
    if hasattr(value, "tolist"):
        return value.tolist()
    return list(value)


def load_detect_model():
    global DETECT_MODEL_INSTANCE
    if DETECT_MODEL_INSTANCE is None:
        DETECT_MODEL_INSTANCE = YOLO(str(DETECT_MODEL_PATH_RESOLVED))
    return DETECT_MODEL_INSTANCE


def load_sam_model():
    global SAM_MODEL_INSTANCE
    if SAM_MODEL_INSTANCE is None:
        SAM_MODEL_INSTANCE = SAM(str(SAM_MODEL_PATH_RESOLVED))
    return SAM_MODEL_INSTANCE


def detection_record_from_xyxy(box: list[float], width: int, height: int, confidence: float, fallback: bool = False) -> dict:
    xyxy = clamp_xyxy(box, width, height)
    bbox = xyxy_to_xywh(xyxy, width, height)
    return {
        "xyxy": xyxy,
        "bbox": bbox,
        "confidence": float(confidence),
        "area": bbox_area(bbox),
        "fallback": bool(fallback),
    }


def extract_detection_boxes(boxes, width: int, height: int) -> list[dict]:
    if boxes is None or len(boxes) == 0:
        return []

    xyxy_values = tensor_to_list(boxes.xyxy)
    conf_values = tensor_to_list(getattr(boxes, "conf", None)) or [1.0] * len(xyxy_values)
    records = []
    for xyxy, conf in zip(xyxy_values, conf_values):
        record = detection_record_from_xyxy(xyxy, width, height, float(conf), fallback=False)
        if record["area"] > 0:
            records.append(record)

    if DETECT_BOX_SELECTION == "largest":
        records.sort(key=lambda item: item["area"], reverse=True)
        records = records[:1]
    elif DETECT_BOX_SELECTION == "confidence":
        records.sort(key=lambda item: item["confidence"], reverse=True)
        records = records[:1]
    else:
        records.sort(key=lambda item: (item["confidence"], item["area"]), reverse=True)

    if MAX_BOXES_PER_IMAGE is not None:
        records = records[:MAX_BOXES_PER_IMAGE]
    return records


def predict_detection_boxes(image_path: Path, width: int, height: int) -> tuple[list[dict], str]:
    model = load_detect_model()
    results = model.predict(
        source=str(image_path),
        conf=DETECT_CONF,
        iou=DETECT_IOU,
        imgsz=DETECT_IMGSZ,
        max_det=DETECT_MAX_DET,
        verbose=False,
        save=False,
    )
    records = extract_detection_boxes(getattr(results[0], "boxes", None), width, height) if results else []
    if records:
        return records, "detect"
    if FALLBACK_TO_FULL_IMAGE_BOX:
        return [detection_record_from_xyxy(full_image_xyxy(width, height), width, height, 0.0, fallback=True)], "full_image_fallback"
    return [], "missing"


def extract_polygons_from_sam_masks(masks, width: int, height: int) -> list[dict]:
    if masks is None or getattr(masks, "xy", None) is None:
        return []

    polygons = []
    for polygon in masks.xy:
        segmentation = []
        for point in polygon:
            x, y = point[:2]
            segmentation.extend([float(x), float(y)])
        segmentation = clamp_polygon(segmentation, width, height)
        area = polygon_area(segmentation)
        area_ratio = area / float(max(width * height, 1))
        if len(segmentation) >= 6 and 0.0005 <= area_ratio <= 0.995:
            polygons.append({
                "segmentation": [segmentation],
                "bbox": bbox_from_segmentation(segmentation),
                "area": area,
            })
    polygons.sort(key=lambda item: item["area"], reverse=True)
    return polygons


def run_sam_with_box(image_path: Path, box_xyxy: list[float]):
    model = load_sam_model()
    prompt_box = [float(v) for v in box_xyxy]
    kwargs = {
        "source": str(image_path),
        "bboxes": [prompt_box],
        "imgsz": SAM_IMGSZ,
        "verbose": False,
        "save": False,
    }
    if SAM_RETINA_MASKS:
        kwargs["retina_masks"] = True
    try:
        return model.predict(**kwargs)
    except TypeError:
        kwargs.pop("retina_masks", None)
        try:
            return model.predict(**kwargs)
        except (TypeError, ValueError):
            kwargs["bboxes"] = prompt_box
            try:
                return model.predict(**kwargs)
            except TypeError:
                return model(str(image_path), bboxes=prompt_box)


def segment_box_with_sam(image_path: Path, box_record: dict, width: int, height: int) -> tuple[dict | None, str]:
    results = run_sam_with_box(image_path, box_record["xyxy"])
    if results is None:
        results = []
    if not isinstance(results, (list, tuple)):
        results = [results]

    polygons = []
    for result in results:
        polygons.extend(extract_polygons_from_sam_masks(getattr(result, "masks", None), width, height))

    if polygons:
        annotation = polygons[0]
        annotation["prompt_bbox"] = box_record["bbox"]
        annotation["detect_confidence"] = box_record["confidence"]
        return annotation, "sam"

    if BOX_FALLBACK_TO_RECTANGLE:
        segmentation = rectangle_segmentation_from_xyxy(box_record["xyxy"], width, height)
        return {
            "segmentation": [segmentation],
            "bbox": bbox_from_segmentation(segmentation),
            "area": polygon_area(segmentation),
            "prompt_bbox": box_record["bbox"],
            "detect_confidence": box_record["confidence"],
        }, "rectangle_fallback"

    return None, "missing"


def predict_yolo_sam_annotations(image_path: Path, width: int, height: int) -> tuple[list[dict], str]:
    box_records, box_source = predict_detection_boxes(image_path, width, height)
    if not box_records:
        return [], "missing"

    annotation_items = []
    for box_index, box_record in enumerate(box_records):
        annotation_data, mask_source = segment_box_with_sam(image_path, box_record, width, height)
        if annotation_data is None:
            continue
        if box_source == "full_image_fallback":
            prediction_source = "full_image_sam" if mask_source == "sam" else "full_image_rectangle_fallback"
        else:
            prediction_source = "detect_sam" if mask_source == "sam" else "detect_rectangle_fallback"
        annotation_data["prediction_source"] = prediction_source
        annotation_data["detection_box_index"] = box_index
        annotation_items.append(annotation_data)

    if not annotation_items:
        return [], "missing"
    sources = {item["prediction_source"] for item in annotation_items}
    return annotation_items, next(iter(sources)) if len(sources) == 1 else "mixed"


def annotation_cache_model_key() -> str:
    return ":".join([
        "yolo_box_prompted_sam",
        str(DETECT_MODEL_PATH_RESOLVED),
        str(SAM_MODEL_PATH_RESOLVED),
        f"detect_conf={DETECT_CONF}",
        f"detect_iou={DETECT_IOU}",
        f"detect_imgsz={DETECT_IMGSZ}",
        f"detect_max_det={DETECT_MAX_DET}",
        f"box_selection={DETECT_BOX_SELECTION}",
        f"max_boxes={MAX_BOXES_PER_IMAGE}",
        f"sam_imgsz={SAM_IMGSZ}",
        f"sam_retina={SAM_RETINA_MASKS}",
        f"full_image_fallback={FALLBACK_TO_FULL_IMAGE_BOX}",
        f"rectangle_fallback={BOX_FALLBACK_TO_RECTANGLE}",
    ])


In [ ]:
def collect_realwaste_images(root_dir: Path) -> list[tuple[str, Path]]:
    image_paths = []
    for class_name in sorted(EXPECTED_CLASSES):
        class_dir = root_dir / class_name
        if not class_dir.exists():
            print(f"Warning: class folder not found: {class_dir}")
            continue
        for image_path in sorted(class_dir.iterdir()):
            if image_path.is_file() and image_path.suffix.lower() in IMAGE_EXTENSIONS:
                image_paths.append((class_name, image_path))
    if MAX_IMAGES is not None:
        image_paths = image_paths[:MAX_IMAGES]
    return image_paths


def prepare_realwaste_annotations(root_dir: Path) -> dict:
    OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
    cache = load_json(CACHE_PATH)
    image_paths = collect_realwaste_images(root_dir)
    if not image_paths:
        raise FileNotFoundError(f"No RealWaste images found under {root_dir}")

    cache_model_key = annotation_cache_model_key()
    print(f"Preparing {len(image_paths)} RealWaste images from {root_dir}")
    print("Pipeline: YOLO detector boxes -> SAM box-prompted masks -> COCO polygon segmentation")
    print(f"Prediction cache: {CACHE_PATH}")
    print(f"Cache model key: {cache_model_key}")

    images = []
    annotations = []
    counts_by_class = {class_name: 0 for class_name in sorted(EXPECTED_CLASSES)}
    predictions_by_source = {}

    for image_index, (class_name, image_path) in enumerate(image_paths, start=1):
        with Image.open(image_path) as image:
            width, height = image.size

        cache_key = image_path.relative_to(root_dir).as_posix()
        cached_prediction = cache.get(cache_key)
        if (
            cached_prediction is not None
            and not FORCE_REGENERATE_ANNOTATIONS
            and cached_prediction.get("model_key") == cache_model_key
            and int(cached_prediction.get("width", width)) == width
            and int(cached_prediction.get("height", height)) == height
        ):
            annotation_items = cached_prediction.get("annotation_data", [])
            prediction_source = cached_prediction.get("prediction_source", "cached")
        else:
            annotation_items, prediction_source = predict_yolo_sam_annotations(image_path, width, height)
            cache[cache_key] = {
                "annotation_data": annotation_items,
                "prediction_source": prediction_source,
                "model_key": cache_model_key,
                "width": width,
                "height": height,
            }

        predictions_by_source[prediction_source] = predictions_by_source.get(prediction_source, 0) + 1

        output_file_name = f"{safe_file_component(class_name)}_{image_path.name.replace(' ', '_')}"
        shutil.copy2(image_path, OUTPUT_DIR / output_file_name)

        image_id = len(images) + 1
        images.append({
            "id": image_id,
            "width": width,
            "height": height,
            "file_name": output_file_name,
            "source_file_name": cache_key,
            "source_class_name": class_name,
        })

        for annotation_data in annotation_items:
            if not annotation_data.get("segmentation"):
                continue
            annotations.append({
                "id": len(annotations) + 1,
                "image_id": image_id,
                "category_id": 1,
                "bbox": annotation_data["bbox"],
                "area": annotation_data["area"],
                "segmentation": annotation_data["segmentation"],
                "iscrowd": 0,
                "prompt_bbox": annotation_data.get("prompt_bbox"),
                "detect_confidence": annotation_data.get("detect_confidence"),
                "prediction_source": annotation_data.get("prediction_source", prediction_source),
                "detection_box_index": annotation_data.get("detection_box_index"),
            })

        counts_by_class[class_name] += 1

        if image_index % SAVE_CACHE_EVERY == 0:
            save_json(cache, CACHE_PATH)
        if image_index % 100 == 0:
            print(f"Prepared {image_index}/{len(image_paths)} images; annotations: {len(annotations)}")

    save_json(cache, CACHE_PATH)
    payload = {
        "images": images,
        "annotations": annotations,
        "categories": CATEGORIES,
        "info": {
            "source": "RealWaste",
            "annotation_task": "yolo_box_prompted_sam_segmentation",
            "detect_model_path": str(DETECT_MODEL_PATH_RESOLVED),
            "detect_conf": DETECT_CONF,
            "detect_iou": DETECT_IOU,
            "detect_imgsz": DETECT_IMGSZ,
            "detect_max_det": DETECT_MAX_DET,
            "detect_box_selection": DETECT_BOX_SELECTION,
            "max_boxes_per_image": MAX_BOXES_PER_IMAGE,
            "sam_model_path": str(SAM_MODEL_PATH_RESOLVED),
            "sam_imgsz": SAM_IMGSZ,
            "sam_retina_masks": SAM_RETINA_MASKS,
            "fallback_to_full_image_box": FALLBACK_TO_FULL_IMAGE_BOX,
            "box_fallback_to_rectangle": BOX_FALLBACK_TO_RECTANGLE,
            "prediction_cache": str(CACHE_PATH),
            "predictions_by_source": predictions_by_source,
        },
    }
    save_json(payload, ANNOTATIONS_PATH)

    summary = {
        "realwaste_root": str(root_dir),
        "output_dir": str(OUTPUT_DIR),
        "annotations_path": str(ANNOTATIONS_PATH),
        "annotation_task": "yolo_box_prompted_sam_segmentation",
        "detect_model_path": str(DETECT_MODEL_PATH_RESOLVED),
        "sam_model_path": str(SAM_MODEL_PATH_RESOLVED),
        "images": len(images),
        "annotations": len(annotations),
        "predictions_by_source": predictions_by_source,
        "counts_by_class": counts_by_class,
        "target_categories": CATEGORIES,
    }
    save_json(summary, SUMMARY_PATH)
    return summary


In [ ]:
REALWASTE_ROOT = discover_realwaste_root()
print("REALWASTE_ROOT:", REALWASTE_ROOT)

summary = prepare_realwaste_annotations(REALWASTE_ROOT)
print(json.dumps(summary, indent=2)[:4000])


In [ ]:
def visualize_prepared_samples(samples: int = 6) -> None:
    import matplotlib.pyplot as plt
    import matplotlib.patches as patches
    from matplotlib.patches import Polygon

    payload = load_json(ANNOTATIONS_PATH)
    annotations_by_image = {}
    for ann in payload.get("annotations", []):
        annotations_by_image.setdefault(ann["image_id"], []).append(ann)
    categories = {cat["id"]: cat["name"] for cat in payload.get("categories", [])}
    images = payload.get("images", [])[:samples]
    if not images:
        print("No prepared images to visualize.")
        return

    cols = min(3, len(images))
    rows = (len(images) + cols - 1) // cols
    fig, axes = plt.subplots(rows, cols, figsize=(5 * cols, 5 * rows), squeeze=False)
    for ax in axes.flat:
        ax.axis("off")

    colors = ["yellow", "cyan", "lime", "magenta", "orange", "red"]
    for ax, image_info in zip(axes.flat, images):
        image_path = OUTPUT_DIR / image_info["file_name"]
        anns = annotations_by_image.get(image_info["id"], [])
        image = Image.open(image_path)
        ax.imshow(image)
        ax.set_title(image_info["file_name"], fontsize=8)
        for ann_index, ann in enumerate(anns):
            color = colors[ann_index % len(colors)]
            label = categories.get(ann["category_id"], str(ann["category_id"]))
            label_x, label_y = ann["bbox"][0], ann["bbox"][1]
            for segmentation in ann.get("segmentation", []):
                points = list(zip(segmentation[0::2], segmentation[1::2]))
                if not points:
                    continue
                ax.add_patch(Polygon(points, closed=True, fill=False, edgecolor=color, linewidth=1.8))
                label_x, label_y = points[0]
            if ann.get("prompt_bbox"):
                x, y, w, h = ann["prompt_bbox"]
                ax.add_patch(patches.Rectangle((x, y), w, h, fill=False, edgecolor="white", linewidth=1.0, linestyle="--"))
            ax.text(
                label_x,
                label_y,
                label,
                color="black",
                fontsize=8,
                bbox={"facecolor": color, "edgecolor": "none", "pad": 1},
            )

    plt.tight_layout()
    plt.show()


visualize_prepared_samples(samples=6)


In [ ]:
ZIP_OUTPUT_DIR = WORKING_DIR / "artifacts"
ZIP_OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

zip_base_path = ZIP_OUTPUT_DIR / "realwaste_yolo_sam_segmentations"
zip_path = shutil.make_archive(
    base_name=str(zip_base_path),
    format="zip",
    root_dir=OUTPUT_DIR.parent,
    base_dir=OUTPUT_DIR.name,
)

print("RealWaste result folder:", OUTPUT_DIR)
print("Zip file ready:", zip_path)
